This script recursively finds all data folders inside valid order*/ directories, renames any .csv files inside them to match their parent folder name (e.g., order12a.csv), and copies each renamed file into a single "raw output" directory.

A log (Renamed_Files_Log.csv) is written to the desktop summarizing all rename and copy actions.
Set DRY_RUN=True to preview actions without changing files.

In [ ]:
# Rename CSVs in order*/data and copy to "raw output"

import os, re, shutil, pandas as pd

# === CONFIG ===
base_dir     = '/path/to/Lexical_Decisions_Run_2025'
desktop_dir  = os.path.expanduser('~/Desktop')
log_csv_path = os.path.join(desktop_dir, 'Renamed_Files_Log.csv')

DATA_DIR_NAME = 'data'
CSV_EXTS      = {'.csv'}
DRY_RUN       = False

ONLY_ORDER_DIRS = True
ORDER_DIR_REGEX = re.compile(r'^order(?:[1-9]\d?|[1-6]\d|7[0-2])[a-h](?:B)?$', re.IGNORECASE)

raw_output_dir = os.path.join(base_dir, 'raw output')
os.makedirs(raw_output_dir, exist_ok=True)

# === HELPERS ===
def unique_name_in_dir(directory, desired_name):
    """Make a non-colliding name."""
    base, ext = os.path.splitext(desired_name)
    candidate, n = desired_name, 1
    while os.path.exists(os.path.join(directory, candidate)):
        candidate = f"{base}_{n}{ext}"
        n += 1
    return candidate

def is_csv(filename):
    """Is CSV?"""
    return os.path.splitext(filename)[1].lower() in CSV_EXTS

# === RUN ===
rename_log, found_data_dirs, processed_files = [], [], 0

for root, dirs, files in os.walk(base_dir):
    if os.path.basename(root).lower() != DATA_DIR_NAME:
        continue
    parent = os.path.basename(os.path.dirname(root))
    if ONLY_ORDER_DIRS and not ORDER_DIR_REGEX.match(parent):
        continue

    found_data_dirs.append(root)
    csvs = sorted([f for f in files if is_csv(f)])
    if not csvs:
        continue

    used_names = set()
    for f in csvs:
        old = os.path.join(root, f)
        new = f"{parent}.csv"
        base_n, ext_n = os.path.splitext(new)
        k = 1
        while new in used_names or os.path.exists(os.path.join(root, new)):
            new = f"{base_n}_{k}{ext_n}"
            k += 1
        used_names.add(new)

        new_path = os.path.join(root, new)
        copied, action = '', 'noop'

        # Rename
        if not DRY_RUN and old != new_path:
            os.rename(old, new_path)
            action = 'renamed'
        elif DRY_RUN and old != new_path:
            action = 'DRY_RUN: rename'

        # Copy
        dest_name = unique_name_in_dir(raw_output_dir, os.path.basename(new_path))
        dest_path = os.path.join(raw_output_dir, dest_name)
        if not DRY_RUN:
            shutil.copy2(new_path, dest_path)
            copied = dest_path
        else:
            copied = f"DRY_RUN: copy -> {dest_path}"

        processed_files += 1
        rename_log.append({
            'Folder': parent,
            'Original Filename': f,
            'New Filename': os.path.basename(new_path),
            'Old Path': old,
            'New Path': new_path,
            'Copied To': copied,
            'Action': action,
            'Data Dir': root
        })

# === SUMMARY ===
print(f"Dirs: {len(found_data_dirs)} | Files: {processed_files}")

# === LOG ===
if rename_log:
    pd.DataFrame(rename_log).to_csv(log_csv_path, index=False)
    print(f"Log: {log_csv_path}")
else:
    print("No files.")


Parses experiment CSVs to extract post-practice trials, separating paired word trials and image trials.
Saves them as _responses.csv and _image_trials.csv in specified output folders.

In [ ]:
# Extract trial responses after practice; save paired word trials and image trials

import os, re, pandas as pd

# --- DETECTORS ---
def is_trial_fixation_cross(stimulus):
    if pd.notna(stimulus):
        s = ''.join(stimulus.replace('"', '').replace("'", '').split())
        return s == '<pstyle=font-size:48px;><br><br>+<br><br></p>'
    return False

def is_green_fixation_cross(stimulus):
    return pd.notna(stimulus) and '+' in stimulus and 'color:#1a851a' in stimulus

def is_instruction_screen(stimulus):
    return pd.notna(stimulus) and (
        'Press enter to begin the trials' in stimulus or
        'Please look at the fixation cross' in stimulus
    )

# --- PROCESS ONE CSV ---
def process_csv(file_path, output_dir, image_output_dir):
    filename = os.path.basename(file_path)
    print(f"Processing: {filename}")
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except Exception as e:
        print(f"Read error: {e}")
        return

    required = ['trial_type', 'trial_index', 'rt', 'stimulus', 'response']
    if not all(c in df.columns for c in required):
        print("Missing columns; skip.")
        return

    practice_end_pattern = r'The practice trials have been completed.*?Again, please respond as quickly and accurately as possible'
    idxs = df[df['stimulus'].str.contains(practice_end_pattern, regex=True, na=False)].index
    if len(idxs) == 0:
        print("No practice end; skip.")
        return
    practice_end_index = idxs[0]

    experiment_df = df.loc[practice_end_index + 1:].reset_index(drop=True)

    current_trial = 0
    in_trial = False
    trial_data, image_trial_data = [], []
    response_collected = False
    is_image_trial = False
    response_phase = 'NO RESPONSE'
    response_rt = None
    response_response = None
    stimulus_content = None

    for _, row in experiment_df.iterrows():
        stimulus = row['stimulus']
        response = row['response']
        rt = row['rt']

        if is_instruction_screen(stimulus):
            continue

        if not in_trial:
            if is_trial_fixation_cross(stimulus):
                in_trial = True
                current_trial += 1
                stimulus_content = None
                response_collected = False
                is_image_trial = False
                response_phase = 'NO RESPONSE'
                response_rt = None
                response_response = None
            continue

        if is_green_fixation_cross(stimulus):
            if in_trial:
                info = {
                    'Trial': current_trial,
                    'Response': response_response if response_collected else None,
                    'RT': response_rt if response_collected else None,
                    'Stimulus': stimulus_content,
                    'Response Time Category': response_phase
                }
                (image_trial_data if is_image_trial else trial_data).append(info)
                in_trial = False
            continue

        if in_trial:
            if pd.notna(stimulus) and ('<img' in stimulus or 'img' in stimulus):
                is_image_trial = True

            phase = None
            if pd.notna(stimulus):
                if '###' in stimulus:
                    phase = 'MASK'
                elif '+' in stimulus and 'color:red' in stimulus:
                    phase = 'FIXATION'
                elif '<p style=' in stimulus:
                    phase = 'WORD'
                    m = re.search(r'<p style=font-size:48px;><br><br>\s*(.*?)\s*<br><br></p>', stimulus)
                    if m:
                        stimulus_content = m.group(1).strip()

            if not response_collected and pd.notna(response) and pd.notna(rt):
                if not is_green_fixation_cross(stimulus):
                    response_collected = True
                    response_rt = rt
                    response_response = response
                    response_phase = phase

    if in_trial:
        info = {
            'Trial': current_trial,
            'Response': response_response if response_collected else None,
            'RT': response_rt if response_collected else None,
            'Stimulus': stimulus_content,
            'Response Time Category': response_phase
        }
        (image_trial_data if is_image_trial else trial_data).append(info)

    if trial_data:
        paired = []
        for i in range(0, len(trial_data), 2):
            t1 = trial_data[i]
            t2 = trial_data[i+1] if i+1 < len(trial_data) else None
            paired.append({
                'Trial1': t1['Trial'],
                'Response1': t1['Response'],
                'RT1': t1['RT'],
                'Stimulus1': t1['Stimulus'],
                'Response Time Category1': t1['Response Time Category'],
                'Trial2': t2['Trial'] if t2 else None,
                'Response2': t2['Response'] if t2 else None,
                'RT2': t2['RT'] if t2 else None,
                'Stimulus2': t2['Stimulus'] if t2 else None,
                'Response Time Category2': t2['Response Time Category'] if t2 else None
            })
        out = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}_responses.csv")
        pd.DataFrame(paired).to_csv(out, index=False)

    if image_trial_data:
        out_img = os.path.join(image_output_dir, f"{os.path.splitext(filename)[0]}_image_trials.csv")
        pd.DataFrame(image_trial_data).to_csv(out_img, index=False)

# --- FIND DATA/PROCESS ---
def search_and_process_data(input_dir, output_dir, image_output_dir):
    for root, dirs, _ in os.walk(input_dir):
        if 'data' in dirs:
            data_folder = os.path.join(root, 'data')
            for fn in os.listdir(data_folder):
                if fn.lower().endswith('.csv'):
                    process_csv(os.path.join(data_folder, fn), output_dir, image_output_dir)

# --- PATHS ---
input_dir = '/path/to/results/Lexical_Decisions_Run_2025'
output_dir = '/path/to/results/SessionFiles'
image_output_dir = '/path/to/results/catchimages'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_output_dir, exist_ok=True)

# --- GO ---
search_and_process_data(input_dir, output_dir, image_output_dir)


Checks the first 10 trials of each session CSV to calculate practice accuracy.
Highlights rows with 0% accuracy in red.

In [ ]:
# Assess first-10/practice accuracy; highlight only 0% rows

import pandas as pd, glob, os

# --- CONFIG ---
directory = '/path/to/results/SessionFiles'
files = glob.glob(os.path.join(directory, '*.csv'))

default_key  = {0:'z',1:'z',2:'m',3:'z',4:'m',5:'m',6:'z',7:'m',8:'z',9:'m'}
reversed_key = {0:'m',1:'m',2:'z',3:'m',4:'z',5:'z',6:'m',7:'z',8:'m',9:'z'}

rows = []

for file in files:
    try:
        df = pd.read_csv(file)
        df.columns = df.columns.str.strip()

        # pick response column
        if   'Response' in df.columns:  resp = 'Response'
        elif 'response' in df.columns:  resp = 'response'
        elif 'Response1' in df.columns: resp = 'Response1'
        elif 'Response2' in df.columns: resp = 'Response2'
        else:
            cands = [c for c in df.columns if 'response' in c.lower() and 'time' not in c.lower()]
            if not cands: raise KeyError("No response column.")
            resp = cands[0]

        # compute accuracy on first 10
        part = df.head(10).copy()
        key  = reversed_key if 'B' in os.path.basename(file) else default_key
        if not part.empty:
            part['CorrectResponse'] = part.index.map(key)
            part['IsCorrect'] = part[resp] == part['CorrectResponse']
            acc = 100 * part['IsCorrect'].mean()
            rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': acc})
        else:
            rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': 'No Data'})

    except Exception as e:
        rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': f'Error: {e}'})

results = pd.DataFrame(rows)

def highlight_fail_all(row):
    v = row.get('Accuracy (%)')
    if isinstance(v, (int, float, float)) and v <= 0:
        return ['background-color: red'] * len(row)
    return [''] * len(row)

results.style.apply(highlight_fail_all, axis=1)


Scans accuracy CSVs and automatically reverses “yes/no” accuracy values in files with <20% accuracy (participants who misread instructions).
Reports both original and corrected accuracy, flags flipped files, and highlights 0% results in red.

In [ ]:
# Auto-flip z/m for low-accuracy files; report final accuracy

import pandas as pd, glob, os

# --- PATHS ---
directory = '/path/to/results/Accuracy'
files = glob.glob(os.path.join(directory, '*.csv'))

# --- HELPERS ---
def count_yes(s):   return s.astype('string').str.lower().eq('yes').sum()
def count_tot(s):   return s.notna().sum()
def acc(df):
    c1, c2 = count_yes(df['Accuracy1']), count_yes(df['Accuracy2'])
    t1, t2 = count_tot(df['Accuracy1']), count_tot(df['Accuracy2'])
    tot = t1 + t2
    return (100 * (c1 + c2) / tot) if tot else 0.0, (c1 + c2), tot

# --- RUN ---
rows, flipped = [], []
for path in files:
    fname = os.path.basename(path)
    try:
        df = pd.read_csv(path)
        if not {'Accuracy1','Accuracy2'}.issubset(df.columns):
            rows.append({'Filename': fname, 'Reversed?': 'no',
                         'Original Accuracy (%)': 'Columns Missing',
                         'Final Accuracy (%)': 'Columns Missing',
                         'Total Correct': '', 'Total Trials': ''})
            continue

        for col in ('Accuracy1','Accuracy2'):
            df[col] = df[col].astype('string').str.strip()

        orig_acc, orig_corr, orig_tot = acc(df)

        rev = False
        fin_acc, fin_corr, fin_tot = orig_acc, orig_corr, orig_tot
        if isinstance(orig_acc, (int, float)) and orig_tot > 0 and orig_acc < 20:
            swap = {'yes':'no','no':'yes'}
            for col in ('Accuracy1','Accuracy2'):
                low = df[col].str.lower()
                df[col] = low.map(swap).fillna(low).astype('string')
            fin_acc, fin_corr, fin_tot = acc(df)
            rev = True
            flipped.append(fname)

        rows.append({
            'Filename': fname,
            'Reversed?': 'yes' if rev else 'no',
            'Original Accuracy (%)': round(orig_acc, 2) if isinstance(orig_acc, (int, float)) else orig_acc,
            'Final Accuracy (%)': round(fin_acc, 2) if isinstance(fin_acc, (int, float)) else fin_acc,
            'Total Correct': int(fin_corr) if isinstance(fin_corr, (int, float)) else '',
            'Total Trials': int(fin_tot) if isinstance(fin_tot, (int, float)) else ''
        })

    except Exception as e:
        rows.append({'Filename': fname, 'Reversed?': 'no',
                     'Original Accuracy (%)': f'Error: {e}',
                     'Final Accuracy (%)': f'Error: {e}',
                     'Total Correct': '', 'Total Trials': ''})

results = pd.DataFrame(rows)

# --- STYLE ---
def highlight_zero(row):
    v = row.get('Final Accuracy (%)')
    if isinstance(v, (int, float)) and float(v) == 0.0:
        return ['background-color: red'] * len(row)
    return [''] * len(row)

# --- LOG ---
print("Flipped:" if flipped else "No reversals.")
for f in flipped: print(f" - {f}")

results.style.apply(highlight_zero, axis=1)


Calculates each file’s overall accuracy from Accuracy1 and Accuracy2 columns.
Highlights rows with accuracy below 80% for quick visual screening.

In [ ]:
# Compute per-file overall accuracy; highlight <80%

import pandas as pd, glob, os

# --- PATHS ---
directory = '/path/to/results/Accuracy'
files = glob.glob(os.path.join(directory, '*.csv'))

# --- RUN ---
rows = []
for file in files:
    try:
        df = pd.read_csv(file)
        if {'Accuracy1','Accuracy2'}.issubset(df.columns):
            c1 = df['Accuracy1'].astype(str).str.lower().eq('yes').sum()
            c2 = df['Accuracy2'].astype(str).str.lower().eq('yes').sum()
            t1 = df['Accuracy1'].notna().sum()
            t2 = df['Accuracy2'].notna().sum()
            tot = t1 + t2
            acc = 100 * (c1 + c2) / tot if tot else 0
            rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': acc})
        else:
            rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': 'Columns Missing'})
    except Exception as e:
        rows.append({'Filename': os.path.basename(file), 'Accuracy (%)': f'Error: {e}'})

results = pd.DataFrame(rows)

# --- STYLE ---
def highlight_low(row):
    v = row.get('Accuracy (%)')
    if isinstance(v, (int, float)) and v < 80:
        return ['background-color: red'] * len(row)
    return [''] * len(row)

results.style.apply(highlight_low, axis=1)


Checks accuracy CSVs and reverses yes/no labels when overall accuracy is below 20% (participant key errors).
Outputs both original and corrected accuracy, lists flipped files, and highlights 0% results in red.

In [ ]:
# Auto-flip z/m when original accuracy <20%; show final accuracy

import pandas as pd, glob, os

# --- PATHS ---
directory = '/path/to/results/Accuracy'
files = glob.glob(os.path.join(directory, '*.csv'))

rows, flipped = [], []

def count_yes(s): return s.astype('string').str.lower().eq('yes').sum()
def count_tot(s): return s.notna().sum()
def compute_acc(df):
    c1, c2 = count_yes(df['Accuracy1']), count_yes(df['Accuracy2'])
    t1, t2 = count_tot(df['Accuracy1']), count_tot(df['Accuracy2'])
    tot = t1 + t2
    return (100 * (c1 + c2) / tot) if tot else 0.0, (c1 + c2), tot

for path in files:
    fname = os.path.basename(path)
    try:
        df = pd.read_csv(path)
        if not {'Accuracy1','Accuracy2'}.issubset(df.columns):
            rows.append({'Filename': fname, 'Reversed?': 'no',
                         'Original Accuracy (%)': 'Columns Missing',
                         'Final Accuracy (%)': 'Columns Missing',
                         'Total Correct': '', 'Total Trials': ''})
            continue

        for col in ('Accuracy1','Accuracy2'):
            df[col] = df[col].astype('string').str.strip()

        orig_acc, orig_corr, orig_tot = compute_acc(df)

        rev = False
        fin_acc, fin_corr, fin_tot = orig_acc, orig_corr, orig_tot
        if isinstance(orig_acc, (int, float)) and orig_tot > 0 and orig_acc < 20:
            swap = {'yes':'no', 'no':'yes'}
            for col in ('Accuracy1','Accuracy2'):
                low = df[col].str.lower()
                df[col] = low.map(swap).fillna(low).astype('string')
            fin_acc, fin_corr, fin_tot = compute_acc(df)
            rev = True
            flipped.append(fname)

        rows.append({
            'Filename': fname,
            'Reversed?': 'yes' if rev else 'no',
            'Original Accuracy (%)': round(orig_acc, 2) if isinstance(orig_acc, (int, float)) else orig_acc,
            'Final Accuracy (%)': round(fin_acc, 2) if isinstance(fin_acc, (int, float)) else fin_acc,
            'Total Correct': int(fin_corr) if isinstance(fin_corr, (int, float)) else '',
            'Total Trials': int(fin_tot) if isinstance(fin_tot, (int, float)) else ''
        })

    except Exception as e:
        rows.append({'Filename': fname, 'Reversed?': 'no',
                     'Original Accuracy (%)': f'Error: {e}',
                     'Final Accuracy (%)': f'Error: {e}',
                     'Total Correct': '', 'Total Trials': ''})

results = pd.DataFrame(rows)

# --- STYLE ---
def highlight_zero(row):
    v = row.get('Final Accuracy (%)')
    if isinstance(v, (int, float)) and float(v) == 0.0:
        return ['background-color: red'] * len(row)
    return [''] * len(row)

# --- LOG ---
print("Flipped:" if flipped else "No reversals.")
for f in flipped: print(f" - {f}")

results.style.apply(highlight_zero, axis=1)


Checks the first 10 catch-image trials per file, flags low-accuracy (<80%) and PF (0–20%), notes empty data/ folders, and writes AccuracyFail_Catch.csv + accuracyPF_catch.csv to your Desktop.
Use by setting base_dir to your run folder; run the cell to generate the fail/PF reports (optional notebook view highlights low rows)

In [ ]:
# Catch-image first-10 accuracy; flag fails/PF; note empty data/ folders

import os, glob, pandas as pd

# --- PATHS ---
base_dir   = '/path/to/Lexical_Decisions_Run_2025'
catch_dir  = os.path.join(base_dir, 'catchimages')
desktop    = os.path.expanduser('~/Desktop')
fail_csv   = os.path.join(desktop, 'AccuracyFail_Catch.csv')
pf_csv     = os.path.join(desktop, 'accuracyPF_catch.csv')

# --- KEYS ---
key_default  = {0:'z',1:'z',2:'m',3:'z',4:'m',5:'m',6:'z',7:'m',8:'z',9:'m'}
key_reversed = {0:'m',1:'m',2:'z',3:'m',4:'z',5:'z',6:'m',7:'z',8:'m',9:'z'}

# --- COLLECT ---
files = glob.glob(os.path.join(catch_dir, '*.csv'))
print(f"Catch files: {len(files)}")

rows = []
for path in files:
    fname = os.path.basename(path)
    try:
        df = pd.read_csv(path)
        sub = df.head(10).copy()
        if sub.empty or 'Response' not in sub.columns:
            rows.append({'Filename': fname, 'Accuracy (%)': 'No Data'})
            continue
        key = key_reversed if 'B' in fname else key_default
        sub['CorrectResponse'] = sub.index.map(key)
        sub['IsCorrect'] = sub['Response'] == sub['CorrectResponse']
        total = len(sub)
        correct = int(sub['IsCorrect'].sum())
        acc = 100 * correct / total if total else 0.0
        rows.append({'Filename': fname, 'Accuracy (%)': round(acc, 2)})
    except Exception as e:
        rows.append({'Filename': fname, 'Accuracy (%)': f'Error: {e}'})

results = pd.DataFrame(rows)

# --- NUMERIC VIEW ---
results_num = results.copy()
results_num['AccuracyNum'] = pd.to_numeric(results_num['Accuracy (%)'], errors='coerce')

# --- FILTERS ---
fails = results_num[results_num['AccuracyNum'].isna() | (results_num['AccuracyNum'] < 80)].copy()
pf    = results_num[results_num['AccuracyNum'].between(0, 20, inclusive='both')].copy()

# --- EMPTY data/ SCAN ---
def find_empty_data_folders(root_dir: str) -> pd.DataFrame:
    out = []
    for root, dirs, files in os.walk(root_dir):
        if os.path.basename(root) == 'data':
            file_count = sum(1 for f in files if os.path.isfile(os.path.join(root, f)))
            if file_count == 0:
                parent = os.path.basename(os.path.dirname(root))
                out.append({'Filename': f'{parent} [EMPTY data/]', 'Accuracy (%)': 'Empty data folder'})
    return pd.DataFrame(out, columns=['Filename', 'Accuracy (%)'])

empty_df = find_empty_data_folders(base_dir)
if not empty_df.empty:
    empty_df['AccuracyNum'] = pd.NA
    fails = pd.concat([fails, empty_df], ignore_index=True)
    print(f"Empty data/: {len(empty_df)}")
else:
    print("Empty data/: 0")

# --- ORDER/OUTPUT ---
def tidy(df):
    if df.empty:
        return df[['Filename', 'Accuracy (%)']] if 'Accuracy (%)' in df.columns else df
    num  = df[df['AccuracyNum'].notna()].copy().sort_values('AccuracyNum')
    nnum = df[df['AccuracyNum'].isna()].copy()
    out = pd.concat([num, nnum], ignore_index=True)
    return out[['Filename', 'Accuracy (%)']]

fails_out = tidy(fails)
pf_out    = tidy(pf)

fails_out.to_csv(fail_csv, index=False)
pf_out.to_csv(pf_csv, index=False)

print(f"FAIL: {len(fails_out)} -> {fail_csv}")
print(f"PF:   {len(pf_out)} -> {pf_csv}")

# --- OPTIONAL (notebooks) ---
try:
    from IPython.display import display
    def hl(row):
        try:
            v = float(row['Accuracy (%)'])
            return ['background-color:#fdd'] * len(row) if v < 80 else [''] * len(row)
        except Exception:
            return ['background-color:#fdd'] * len(row)
    display(results.style.apply(hl, axis=1))
except Exception:
    pass


Adds per-file RT totals (RTTotal1/2, RTSum, RTDiff) using category offsets and computes within-file z-scores, saving results to Accuracy_withRTTotals/.
Use: set IN_DIR/OUT_DIR, then run; each input CSV must have RT1/RT2 and Response Time Category1/2.

In [ ]:
# Add RT totals/z-scores per file; save to output

import os, glob, pandas as pd, numpy as np
from pathlib import Path

# === CONFIG ===
IN_DIR  = Path("/path/to/results/Accuracy")
OUT_DIR = Path("/path/to/results/Accuracy_withRTTotals")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Category → offset (ms)
OFFSET_MAP = {"WORD": 0, "MASK": 500, "FIXATION": 1000}
NULL_CATEGORIES = {"NO RESPONSE", ""}

# Z-score opts
ZSCORE_DDOF = 0
FILL_CONST_Z = None  # set 0.0 to force zeros when std==0

# --- COL LOOKUP ---
def find_col(cols, target):
    norm = {c: "".join(str(c).lower().split()) for c in cols}
    t = "".join(target.lower().split())
    for c, n in norm.items():
        if n == t: return c
    return None

REQ_COLS = ["RT1","RT2","Response Time Category1","Response Time Category2"]

# --- ZSCORE ---
def zscore_within_series(s: pd.Series, ddof: int = 0) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    m, sd = x.mean(skipna=True), x.std(ddof=ddof, skipna=True)
    if pd.isna(sd) or sd == 0:
        out = pd.Series(np.nan, index=x.index, dtype="float64")
        if FILL_CONST_Z is not None: out[:] = FILL_CONST_Z
        return out
    return (x - m) / sd

# --- PROCESS ONE FILE ---
def process_file(csv_path: Path) -> tuple[bool, str]:
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        return False, f"Read error: {e}"

    col_rt1  = find_col(df.columns, "RT1")
    col_rt2  = find_col(df.columns, "RT2")
    col_cat1 = find_col(df.columns, "Response Time Category1")
    col_cat2 = find_col(df.columns, "Response Time Category2")

    missing = [n for n, c in {
        "RT1": col_rt1, "RT2": col_rt2,
        "Response Time Category1": col_cat1,
        "Response Time Category2": col_cat2,
    }.items() if c is None]
    if missing: return False, f"Missing columns: {', '.join(missing)}"

    cat1 = df[col_cat1].astype("string").str.upper().str.strip()
    cat2 = df[col_cat2].astype("string").str.upper().str.strip()

    null_set = {c.upper() for c in NULL_CATEGORIES}
    null1, null2 = cat1.isin(null_set), cat2.isin(null_set)

    off1 = cat1.map(OFFSET_MAP)
    off2 = cat2.map(OFFSET_MAP)

    rt1 = pd.to_numeric(df[col_rt1], errors="coerce")
    rt2 = pd.to_numeric(df[col_rt2], errors="coerce")

    off1[null1] = np.nan; off2[null2] = np.nan
    rt1[null1]  = np.nan; rt2[null2]  = np.nan

    df["RTTotal1"] = rt1 + off1
    df["RTTotal2"] = rt2 + off2
    df["RTSum"]    = df["RTTotal1"] + df["RTTotal2"]
    df["RTDiff"]   = df["RTTotal2"] - df["RTTotal1"]

    df["z_RTTotal1"] = zscore_within_series(df["RTTotal1"], ddof=ZSCORE_DDOF)
    df["z_RTTotal2"] = zscore_within_series(df["RTTotal2"], ddof=ZSCORE_DDOF)
    df["z_RTSum"]    = zscore_within_series(df["RTSum"],    ddof=ZSCORE_DDOF)

    out_path = OUT_DIR / csv_path.name
    try:
        df.to_csv(out_path, index=False)
    except Exception as e:
        return False, f"Write error: {e}"

    unknown1 = cat1[~cat1.isin(OFFSET_MAP.keys()) & ~null1].dropna().unique()
    unknown2 = cat2[~cat2.isin(OFFSET_MAP.keys()) & ~null2].dropna().unique()

    n1, n2 = int(null1.sum()), int(null2.sum())
    null_note = f" (nulled: c1={n1}, c2={n2})" if (n1 or n2) else ""
    warn = f" (unknown → NaN: c1={list(unknown1)}, c2={list(unknown2)})" if (len(unknown1) or len(unknown2)) else ""

    return True, f"OK → {out_path}{null_note}{warn}"

# --- RUN ---
csv_files = sorted(IN_DIR.glob("*.csv"))
print(f"Files: {len(csv_files)} | In: {IN_DIR.name} -> Out: {OUT_DIR.name}")

ok = fail = 0
for f in csv_files:
    success, info = process_file(f)
    ok += int(success); fail += int(not success)
    print(f"{f.name}: {info}")

print(f"\nDone | OK: {ok} | Fail: {fail}")
print(f"Out: {OUT_DIR}")


Builds per-file accuracy summary from Accuracy/*.csv, applies a strict PF-gated z/m swap (PF = 0–20% catch accuracy), and writes Accuracy_Overall_WithSwap.csv.
Generates RerunLD.csv excluding: all Fail stems, PF stems still <80% after swap, and any other files with final accuracy <80%.
Set base_dir (ensure accuracyPF_catch.csv and AccuracyFail_Catch.csv exist on Desktop) and run.

In [ ]:
# Build accuracy summary with strict z/m swap; write rerun list

import os, glob, pandas as pd, numpy as np

# --- PATHS ---
base_dir     = '/path/to/Lexical_Decisions_Run_2025'
accuracy_dir = os.path.join(base_dir, 'Accuracy')
desktop      = os.path.expanduser('~/Desktop')

pf_csv_path   = os.path.join(desktop, 'accuracyPF_catch.csv')      # 0–20% catch accuracy
fail_csv_path = os.path.join(desktop, 'AccuracyFail_Catch.csv')    # <80% catch or errors/No Data
summary_out   = os.path.join(desktop, 'Accuracy_Overall_WithSwap.csv')
rerun_out     = os.path.join(desktop, 'RerunLD.csv')

# --- HELPERS ---
def stem_before_underscore(filename: str) -> str:
    base = os.path.basename(filename)
    return base.split('_', 1)[0].lower()

def safe_count_yes(s: pd.Series) -> int:
    return s.astype('string').str.lower().eq('yes').sum()

def safe_count_total(s: pd.Series) -> int:
    return s.notna().sum()

def is_zm(s: pd.Series) -> pd.Series:
    return s.astype('string').str.strip().str.lower().isin({'z', 'm'})

def load_stem_set(path: str) -> set:
    if not os.path.exists(path): return set()
    df = pd.read_csv(path)
    if 'Filename' not in df.columns: return set()
    return {stem_before_underscore(x) for x in df['Filename'].astype(str).tolist()}

# --- LOAD PF/FAIL STEMS ---
pf_stems   = load_stem_set(pf_csv_path)
fail_stems = load_stem_set(fail_csv_path)

# --- PROCESS ACCURACY FILES ---
rows = []
for fpath in glob.glob(os.path.join(accuracy_dir, '*.csv')):
    fname = os.path.basename(fpath)
    stem  = stem_before_underscore(fname)
    try:
        df = pd.read_csv(fpath)

        if not {'Accuracy1','Accuracy2'}.issubset(df.columns):
            rows.append({'Filename': fname, 'Stem': stem,
                         'OriginalAccuracy': None, 'SwapApplied': False,
                         'FinalAccuracy': None, 'Status': 'Columns Missing'})
            continue

        c1, c2 = safe_count_yes(df['Accuracy1']), safe_count_yes(df['Accuracy2'])
        t1, t2 = safe_count_total(df['Accuracy1']), safe_count_total(df['Accuracy2'])
        tot = int(t1 + t2)
        orig_acc = (100 * (c1 + c2) / tot) if tot else 0.0

        has_responses = {'Response1','Response2'}.issubset(df.columns)
        swap_allowed  = (stem in pf_stems) and has_responses and (tot > 0)

        if swap_allowed:
            acc1 = df['Accuracy1'].astype('string').str.lower().eq('yes')
            acc2 = df['Accuracy2'].astype('string').str.lower().eq('yes')
            zm1  = is_zm(df['Response1'])
            zm2  = is_zm(df['Response2'])
            s1   = np.where(zm1, ~acc1, acc1)
            s2   = np.where(zm2, ~acc2, acc2)
            swapped_correct = int(s1.sum() + s2.sum())
            final_acc = 100 * swapped_correct / tot
            swap_applied, status = True, 'OK'
        else:
            final_acc, swap_applied = orig_acc, False
            status = 'OK' if (stem not in pf_stems) else 'PF no-swap (missing Responses)'

        rows.append({'Filename': fname, 'Stem': stem,
                     'OriginalAccuracy': round(orig_acc, 2) if tot else orig_acc,
                     'SwapApplied': swap_applied,
                     'FinalAccuracy': round(final_acc, 2) if tot else final_acc,
                     'Status': status})

    except Exception as e:
        rows.append({'Filename': fname, 'Stem': stem,
                     'OriginalAccuracy': None, 'SwapApplied': False,
                     'FinalAccuracy': None, 'Status': f'Error: {e}'})

summary_df = pd.DataFrame(rows)

# --- SAVE SUMMARY ---
summary_cols = ['Filename', 'OriginalAccuracy', 'SwapApplied', 'FinalAccuracy', 'Status']
summary_df.to_csv(summary_out, index=False, columns=summary_cols)
print(f"Summary: {summary_out}")

# --- RERUN LIST ---
rerun = []

# 1) All in Fail (by stem)
for st in sorted(fail_stems):
    m = summary_df[summary_df['Stem'] == st]
    if not m.empty:
        r = m.iloc[0]
        rerun.append({'Filename': r['Filename'], 'Reason': 'Catch fail (<80% or error)'})
    else:
        rerun.append({'Filename': st, 'Reason': 'Catch fail (<80% or error)'})

added = set(fail_stems)

# 2) PF and still <80% after swap
pf_after = summary_df[(summary_df['Stem'].isin(pf_stems)) &
                      (pd.to_numeric(summary_df['FinalAccuracy'], errors='coerce') < 80)]
for _, r in pf_after.iterrows():
    if r['Stem'] in added: continue
    rerun.append({'Filename': r['Filename'], 'Reason': 'Still <80% after PF swap (strict)'})
    added.add(r['Stem'])

# 3) Others <80% (not PF or Fail)
others = summary_df[(~summary_df['Stem'].isin(pf_stems)) &
                    (~summary_df['Stem'].isin(fail_stems)) &
                    (pd.to_numeric(summary_df['FinalAccuracy'], errors='coerce') < 80)]
for _, r in others.iterrows():
    if r['Stem'] in added: continue
    rerun.append({'Filename': r['Filename'], 'Reason': 'Overall <80% (no PF or Fail flag)'})
    added.add(r['Stem'])

pd.DataFrame(rerun).to_csv(rerun_out, index=False)
print(f"Rerun:   {rerun_out}")


Combines all Accuracy_withRTTotals/*.csv (falls back to Accuracy/ if needed) into one file, excluding stems listed in RerunLD.csv.
Requires RTTotal1/2, RTSum, RTDiff (if enabled), adds a SourceFile column, and writes All_Accuracy_withRTTotals_COMBINED_filtered.csv to Desktop.
Set BASE_DIR, confirm RerunLD.csv on Desktop, run.

In [ ]:
# Combine Accuracy_withRTTotals CSVs; exclude rerun stems

import os, pandas as pd
from pathlib import Path

# --- CONFIG ---
BASE_DIR     = Path("/path/to/Lexical_Decisions_Run_2025")
IN_DIR       = BASE_DIR / "Accuracy_withRTTotals"
FALLBACK_DIR = BASE_DIR / "Accuracy"
DESKTOP_DIR  = Path.home() / "Desktop"

RERUN_PATH = DESKTOP_DIR / "RerunLD.csv"
OUT_PATH   = DESKTOP_DIR / "All_Accuracy_withRTTotals_COMBINED_filtered.csv"

REQUIRE_RT_TOTALS = True
REQUIRED_COLS = {"RTTotal1", "RTTotal2", "RTSum", "RTDiff"}

# --- HELPERS ---
def stem_before_underscore(name: str) -> str:
    """Stem up to first underscore."""
    return os.path.basename(name).split("_", 1)[0].lower()

def load_excluded_stems(rerun_csv: Path) -> set:
    """Load stems from Rerun list."""
    if not rerun_csv.exists(): return set()
    df = pd.read_csv(rerun_csv)
    if "Filename" not in df.columns: return set()
    return {stem_before_underscore(str(x)) for x in df["Filename"].astype(str)}

def choose_input_dir(preferred: Path, fallback: Path) -> Path:
    """Pick preferred if it has CSVs; else fallback; else error."""
    if preferred.exists() and any(preferred.glob("*.csv")): return preferred
    if fallback.exists() and any(fallback.glob("*.csv")):
        print(f"Using fallback: {fallback}")
        return fallback
    raise FileNotFoundError("No CSVs in preferred or fallback.")

# --- LOAD EXCLUSIONS ---
exclude_stems = load_excluded_stems(RERUN_PATH)
print(f"Exclude: {len(exclude_stems)}")

# --- GATHER ---
IN_USE_DIR = choose_input_dir(IN_DIR, FALLBACK_DIR)
csv_files = sorted(IN_USE_DIR.glob("*.csv"))
print(f"Files: {len(csv_files)} in {IN_USE_DIR}")

# --- COMBINE ---
kept = skipped_excluded = skipped_missing = 0
frames = []

for fpath in csv_files:
    stem = stem_before_underscore(fpath.name)
    if stem in exclude_stems:
        skipped_excluded += 1
        continue
    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        print(f"Read error: {fpath.name} | {e}")
        continue
    if REQUIRE_RT_TOTALS and not REQUIRED_COLS.issubset(df.columns):
        skipped_missing += 1
        continue
    df.insert(0, "SourceFile", fpath.name)
    frames.append(df)
    kept += 1

# --- WRITE ---
if frames:
    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined.to_csv(OUT_PATH, index=False)
    print(f"Wrote {len(combined):,} rows from {kept} files -> {OUT_PATH}")
    print(f"Excluded: {skipped_excluded} | MissingRT: {skipped_missing}")
else:
    print("Nothing to write.")


Filters combined data by RT bounds (500–2000) and both accuracies == “yes”, drops out-of-range/NA as configured, then keeps only participants with ≥20 rows.
Writes All_Accuracy_withRTTotals_COMBINED_FINAL.csv to Desktop.
Run after creating the combined file; tweak LOWER/UPPER/MIN_ROWS if needed.

In [ ]:
# Apply filters; write final CSV

import pandas as pd
from pathlib import Path

# --- CONFIG ---
IN_PATH  = Path.home() / "Desktop" / "All_Accuracy_withRTTotals_COMBINED_filtered.csv"
OUT_PATH = Path.home() / "Desktop" / "All_Accuracy_withRTTotals_COMBINED_FINAL.csv"

LOWER, UPPER = 500, 2000
MIN_ROWS = 20
DROP_NA_AS_OUT_OF_RANGE = True
TREAT_MISSING_ACCURACY_AS_NOT_YES = True

# --- COL FINDER ---
def find_col(cols, target):
    norm = {c: "".join(str(c).lower().split()) for c in cols}
    t = "".join(target.lower().split())
    for c, n in norm.items():
        if n == t: return c
    return None

# --- READ ---
df = pd.read_csv(IN_PATH)

col_rt1 = find_col(df.columns, "RTTotal1")
col_rt2 = find_col(df.columns, "RTTotal2")
col_a1  = find_col(df.columns, "Accuracy1")
col_a2  = find_col(df.columns, "Accuracy2")
col_src = find_col(df.columns, "SourceFile") or find_col(df.columns, "Filename")

missing = [n for n, c in {
    "RTTotal1": col_rt1, "RTTotal2": col_rt2,
    "Accuracy1": col_a1, "Accuracy2": col_a2,
    "SourceFile/Filename": col_src
}.items() if c is None]
if missing:
    raise ValueError(f"Missing: {', '.join(missing)}")

# --- FILTERS ---
rt1 = pd.to_numeric(df[col_rt1], errors="coerce")
rt2 = pd.to_numeric(df[col_rt2], errors="coerce")

v1 = (rt1 >= LOWER) & (rt1 <= UPPER)
v2 = (rt2 >= LOWER) & (rt2 <= UPPER)
if DROP_NA_AS_OUT_OF_RANGE:
    v1 = v1.fillna(False); v2 = v2.fillna(False)

a1 = df[col_a1].astype("string").str.strip().str.lower().eq("yes")
a2 = df[col_a2].astype("string").str.strip().str.lower().eq("yes")
if TREAT_MISSING_ACCURACY_AS_NOT_YES:
    a1 = a1.fillna(False); a2 = a2.fillna(False)

keep = v1 & v2 & a1 & a2
primary = df.loc[keep].copy()

counts = primary[col_src].value_counts()
keepers = set(counts[counts >= MIN_ROWS].index)
final = primary[primary[col_src].isin(keepers)].copy()

# --- SAVE ---
final.to_csv(OUT_PATH, index=False)

# --- SUMMARY ---
print(f"In: {len(df):,} | After RT+Acc: {len(primary):,} | Participants: {len(keepers):,} | Final: {len(final):,}")
print(f"Saved: {OUT_PATH}")


Appends participants with < MIN_ROWS usable trials (after RT/accuracy filters) to RerunLD.csv, preserving schema and avoiding duplicates.
Run after computing removed_stems; script loads existing rerun file (or creates one) and reports how many were added.

In [ ]:
# Add "< MIN_ROWS" candidates to rerun list; keep schema; dedupe

to_add_cols = ["Filename", "Reason"]
to_add_rows = []

if removed_stems:
    for st in sorted(removed_stems):
        name = stem_to_filename.get(st, st)
        to_add_rows.append({
            "Filename": name,
            "Reason": f"Too few usable trials (<{MIN_ROWS}) after RT/accuracy filters"
        })
    to_add_df = pd.DataFrame(to_add_rows, columns=to_add_cols)
else:
    to_add_df = pd.DataFrame(columns=to_add_cols)

print(f"Candidates (<{MIN_ROWS}): {len(removed_stems)}")
if removed_stems:
    print(f"Sample: {sorted(list(removed_stems))[:8]}")

# Load existing rerun (or create); ensure columns
rerun_df = pd.read_csv(RERUN_PATH) if RERUN_PATH.exists() else pd.DataFrame(columns=["Filename", "Reason"])
for c in ["Filename", "Reason"]:
    if c not in rerun_df.columns:
        rerun_df[c] = pd.Series(dtype="object")

# No candidates → return existing as-is
if to_add_df.empty:
    updated = rerun_df.copy()
    print(f"Rerun rows: {len(rerun_df)} | Added: 0")


Builds a RerunLDLinks.csv file containing HTML links for every participant in RerunLD.csv.
Each link points to the participant’s local HTML file (e.g., http://ip.ip.ip.ip/path/.../order12a/order12a.html).
Set BASE_URL to your server path, ensure RerunLD.csv is on Desktop, then run to generate shareable rerun links.

In [ ]:
# Build rerun HTML links from RerunLD.csv

import os, re, pandas as pd
from pathlib import Path

# --- PATHS ---
desktop   = Path.home() / "Desktop"
rerun_in  = desktop / "RerunLD.csv"
rerun_out = desktop / "RerunLDLinks.csv"
BASE_URL  = 'http://ip.ip.ip.ip/path/to/Lexical_Decisions_Run_2025'

# --- LOAD ---
if not rerun_in.exists():
    raise FileNotFoundError(f"Missing: {rerun_in}")
df = pd.read_csv(rerun_in)
if 'Filename' not in df.columns:
    raise ValueError("Missing 'Filename' column.")

# --- LINKS ---
def make_basename(filename: str) -> str:
    """'order10bB_accuracy.csv' -> 'order10bB'."""
    name = os.path.basename(str(filename)).strip()
    name = re.sub(r'_accuracy\.csv$', '', name, flags=re.IGNORECASE)
    return re.sub(r'\.csv$', '', name, flags=re.IGNORECASE)

def make_link(filename: str) -> str:
    b = make_basename(filename)
    return f"{BASE_URL}/{b}/{b}.html"

out = pd.DataFrame({'Link': df['Filename'].apply(make_link)}).drop_duplicates().reset_index(drop=True)

# --- SAVE ---
out.to_csv(rerun_out, index=False)
print(f"Saved: {rerun_out}")


Finds files that were renamed (from Renamed_Files_Log.csv) and, for stems listed in RerunLD.csv, deletes the old filenames so only the new names remain (safe by default with DRY_RUN). Prevents accidentally overriding new runs of files
Writes an audit report OldNames_Matched_ByNew_in_RerunLD.csv showing what would be/was removed.
Set paths, confirm DRY_RUN, run to clean up legacy filenames by stem.

In [ ]:
# Remove old names when new names appear in RerunLD (by stem)

import os, pandas as pd

# --- CONFIG ---
base_dir    = '/path/to/results'               # root containing the .../data folders
desktop     = os.path.expanduser('~/Desktop')
log_csv     = os.path.join(desktop, 'Renamed_Files_Log.csv')
rerun_csv   = os.path.join(desktop, 'RerunLD.csv')
report_csv  = os.path.join(desktop, 'OldNames_Matched_ByNew_in_RerunLD.csv')
DRY_RUN = True
MATCH_BY_STEM  = True
CASE_SENSITIVE = True  # used only if MATCH_BY_STEM=False

# --- HELPERS ---
def load_csv(path): 
    if not os.path.exists(path): raise FileNotFoundError(f"Missing: {path}")
    return pd.read_csv(path)

def stem_before_underscore(name: str) -> str:
    return os.path.basename(str(name)).split('_', 1)[0].lower()

def norm_series(s, case_sensitive=True):
    s = s.astype('string')
    return s if case_sensitive else s.str.lower()

def is_under(path, root):
    try:
        return os.path.commonpath([os.path.realpath(path), os.path.realpath(root)]) == os.path.realpath(root)
    except Exception:
        return False

# --- LOAD ---
log_df = load_csv(log_csv)
need = {'Folder','Original Filename','New Filename','Old Path','New Path'}
missing = need - set(log_df.columns)
if missing: raise ValueError(f"Log missing: {sorted(missing)}")

rerun_df = load_csv(rerun_csv)
if 'Filename' not in rerun_df.columns: raise ValueError("RerunLD needs 'Filename'.")

# --- MATCH (new names vs RerunLD) ---
if MATCH_BY_STEM:
    rerun_stems = {stem_before_underscore(x) for x in rerun_df['Filename'].dropna().astype(str)}
    log_df['_stem'] = log_df['New Filename'].astype('string').map(stem_before_underscore)
    matched_mask = log_df['_stem'].isin(rerun_stems)
else:
    rerun_names = set(norm_series(rerun_df['Filename'].dropna(), case_sensitive=CASE_SENSITIVE))
    log_df['_name'] = norm_series(log_df['New Filename'], case_sensitive=CASE_SENSITIVE)
    matched_mask = log_df['_name'].isin(rerun_names)

matched, unmatched = log_df.loc[matched_mask].copy(), log_df.loc[~matched_mask].copy()

# --- DELETE OLD (or simulate) ---
rows = []
for _, r in matched.iterrows():
    old_name = r['Original Filename']
    old_path = r.get('Old Path', '')
    candidate = old_path if (isinstance(old_path, str) and old_path and os.path.isabs(old_path)) \
               else os.path.join(base_dir, r['Folder'], 'data', old_name)

    status, note = 'not found', ''
    if not is_under(candidate, base_dir):
        status, note = 'skipped', 'outside base_dir'
    elif not os.path.exists(candidate):
        status, note = 'not found', 'missing'
    else:
        if DRY_RUN:
            status = 'DRY_RUN: would delete'
        else:
            try:
                os.remove(candidate)
                status = 'deleted'
            except Exception as e:
                status, note = 'error', str(e)

    rows.append({
        'Folder': r['Folder'],
        'Old Filename': old_name,
        'Old Path Checked': candidate,
        'New Filename (matched)': r['New Filename'],
        'New Path (info)': r.get('New Path', ''),
        'Action': status,
        'Notes': note
    })

# include unmatched in report
for _, r in unmatched.iterrows():
    rows.append({
        'Folder': r['Folder'],
        'Old Filename': r['Original Filename'],
        'Old Path Checked': os.path.join(base_dir, r['Folder'], 'data', r['Original Filename']),
        'New Filename (matched)': '',
        'New Path (info)': r.get('New Path', ''),
        'Action': 'excluded',
        'Notes': 'new name not in RerunLD'
    })

# --- REPORT ---
pd.DataFrame(rows).to_csv(report_csv, index=False)
print(f"Matched: {matched.shape[0]} | Report: {report_csv} | DRY_RUN={DRY_RUN}")


Fits several mixed-effects models (SUM and log_SUM ~ true_count_Condition or Condition) with random intercepts by participant, prints model summaries, and reports approximate marginal/conditional R².
Point to your CSV (must include participant, Condition, SUM, and predictors), run to evaluate effects and variance explained.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# Load data
# Assuming the .mat file was converted or exported to a CSV
# Example: ExampleDataset.csv
ExampleDataset = pd.read_csv('/path/to/data')

# Prep categorical variables
ExampleDataset['participant'] = ExampleDataset['participant'].astype('category')
ExampleDataset['Condition'] = ExampleDataset['Condition'].astype('category')

# Add log_SUM but keep SUM
ExampleDataset['log_SUM'] = np.log(ExampleDataset['SUM'])

# Model specifications
models = [
    'SUM ~ true_count_Condition',
    'log_SUM ~ true_count_Condition',
    'SUM ~ Condition',
    'log_SUM ~ Condition'
]

# Random effects specification
random_effects_simple = '1|participant'
random_effects_extended = '1|participant'

# Run models
for i, model in enumerate(models):
    print(f'\nRunning Model {i + 1}: {model}')
    
    # Determine random effects structure
    if 'true_count_Condition' in model:
        random_formula = '1'
    elif 'Condition' in model and '+' not in model:
        random_formula = '1 + Sim + SupFreq + SubFreq + SupSen + SubSen + SubMeanSim + LEN1 + LEN2'
    else:
        random_formula = '1'

    # Build mixed model
    md = mixedlm(model, ExampleDataset, groups=ExampleDataset['participant'])
    mdf = md.fit()
    print(mdf.summary())
    
    # Calculate R²
    fitted_values = mdf.fittedvalues
    residuals = ExampleDataset[model.split('~')[0].strip()] - fitted_values
    fixed_effect_variance = np.var(fitted_values)
    residual_variance = np.var(residuals)

    # Random effects variance approximation
    random_effect_variance = sum([v for v in mdf.cov_re.values.flatten()])

    marginal_r2 = fixed_effect_variance / (fixed_effect_variance + random_effect_variance + residual_variance)
    conditional_r2 = (fixed_effect_variance + random_effect_variance) / (fixed_effect_variance + random_effect_variance + residual_variance)

    print(f'Marginal R² (fixed effects only): {marginal_r2:.3f}')
    print(f'Conditional R² (fixed and random effects): {conditional_r2:.3f}')


Other plots and analyses are below

In [ ]:
# %pip install pandas statsmodels matplotlib seaborn numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

# === Load data ===
csv_path = Path("/path/")
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Ensure participant column exists (rename common variants if needed)
if "participant" not in df.columns:
    for alt in ["Participant", "participant_id", "ParticipantID", "subject", "Subject"]:
        if alt in df.columns:
            df = df.rename(columns={alt: "participant"})
            break
if "participant" not in df.columns:
    raise ValueError("No 'participant' column found in this CSV.")

# Ensure numeric types
df["SUM"] = pd.to_numeric(df["SUM"], errors="coerce")
df["true_count_Condition"] = pd.to_numeric(df["true_count_Condition"], errors="coerce")
df = df.dropna(subset=["SUM", "true_count_Condition", "participant"]).copy()

# === Mixed model on SUM (ML to mirror MATLAB tables) ===
md = smf.mixedlm("SUM ~ true_count_Condition", df, groups=df["participant"])
mdf = md.fit(reml=False)  # ML
print(mdf.summary())

# === Variance components & R^2 (Nakagawa & Schielzeth) ===
X = mdf.model.exog
beta = mdf.fe_params.values        # [Intercept, slope]
fixed_linpred = X @ beta           # Xβ on SUM scale
var_fixed = np.var(fixed_linpred, ddof=1)
var_u = float(mdf.cov_re.iloc[0, 0])   # random-intercept variance
sigma2 = float(mdf.scale)              # residual variance

R2_marginal = var_fixed / (var_fixed + var_u + sigma2)
R2_conditional = (var_fixed + var_u) / (var_fixed + var_u + sigma2)

print("\n--- Variance components ---")
print(f"Var(Xβ) [fixed]: {var_fixed:.6f}")
print(f"Var(u)  [random intercepts]: {var_u:.6f}")
print(f"σ²      [residual]: {sigma2:.6f}")
print(f"Marginal R² (fixed only): {R2_marginal:.3f}")
print(f"Conditional R² (fixed + random): {R2_conditional:.3f}")

# Random intercept per participant
re = mdf.random_effects
def get_re(pid):
    d = re.get(pid, {})
    return float(d.get("Intercept", d.get("Group", 0.0)) or 0.0)

df["rand_intercept"] = df["participant"].map(get_re).fillna(0.0)

# Partial residuals on SUM scale (remove random intercepts)
df["SUM_partial"] = df["SUM"] - df["rand_intercept"]

# === Plot A: partial residuals + bin means ± SE + fixed-effect line (SUM scale) ===
plt.figure(figsize=(9,6))
sns.scatterplot(
    x="true_count_Condition", y="SUM_partial",
    data=df, alpha=0.15, s=15, linewidth=0, label="Partial residuals"
)

gb = (df.groupby("true_count_Condition", as_index=False)["SUM_partial"]
        .agg(["mean","count","std"]).reset_index())
gb.columns = ["true_count_Condition","mean","count","std"]
gb["se"] = gb["std"] / np.sqrt(gb["count"].clip(lower=1))
plt.errorbar(gb["true_count_Condition"], gb["mean"], yerr=gb["se"],
             fmt="o", elinewidth=1, capsize=2, label="Bin mean ± SE")

intercept = float(mdf.fe_params["Intercept"])
slope     = float(mdf.fe_params["true_count_Condition"])
pval      = float(mdf.pvalues["true_count_Condition"])

xgrid = np.linspace(df["true_count_Condition"].min(), df["true_count_Condition"].max(), 200)
plt.plot(xgrid, intercept + slope*xgrid, color="red", linewidth=2.5, label="Fixed-effect line")

plt.xlabel("true_count_Condition")
plt.ylabel("SUM (participant intercepts removed)")
plt.title("Fixed effect after removing participant intercepts\n"
          f"Marginal R² ≈ {R2_marginal:.3f}, slope={slope:.6g}, p={pval:.4g}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# === Plot B: Observed vs CONDITIONAL prediction (shows conditional R² visually) ===
df["fixed_only_pred"]   = fixed_linpred
df["conditional_pred"]  = df["fixed_only_pred"] + df["rand_intercept"]

plt.figure(figsize=(6,6))
plt.scatter(df["conditional_pred"], df["SUM"], alpha=0.15, s=10)
lims = [min(df["conditional_pred"].min(), df["SUM"].min()),
        max(df["conditional_pred"].max(), df["SUM"].max())]
plt.plot(lims, lims, 'r--', lw=2)
plt.xlabel("Conditional prediction (fixed + random)")
plt.ylabel("Observed SUM")
plt.title(f"Observed vs Conditional Predictions\nConditional R² ≈ {R2_conditional:.3f}")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Compute condition-level summary ---
cond_stats = (
    df.groupby("Condition")["SUM_partial"]
      .agg(["mean", "count", "std"])
      .reset_index()
)
cond_stats["se"] = cond_stats["std"] / np.sqrt(cond_stats["count"].clip(lower=1))

# --- Bar plot (zoomed y-axis) ---
plt.figure(figsize=(6,6))

bars = plt.bar(
    cond_stats["Condition"],
    cond_stats["mean"],
    yerr=cond_stats["se"],
    capsize=6,
    color=["#4C72B0", "#55A868"],  # blue and green
    alpha=0.8
)

plt.ylabel("Mean SUM (random intercepts removed)")
plt.xlabel("Condition")
plt.title("Condition averages: SUP-SUB vs SUB-SUP (zoomed)")
plt.ylim(1440, 1480)  # <- zoom here
plt.grid(axis="y", alpha=0.3)

# Annotate values above bars
for bar, mean in zip(bars, cond_stats["mean"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{mean:.1f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# %pip install statsmodels  # uncomment if needed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy import stats
import statsmodels.formula.api as smf
import glob, os, warnings
import statsmodels.api as sm  # NEW: for WLS on bin means


# =========================
# Font: Roboto Condensed
# =========================
def use_roboto_condensed():
    search_dirs = [
        os.path.expanduser("~/Library/Fonts"),
        "/Library/Fonts",
        os.path.expanduser("~/Library/Application Support/Microsoft/Fonts"),
    ]
    ttf_paths = []
    for d in search_dirs:
        try:
            ttf_paths.extend(glob.glob(os.path.join(d, "*Roboto*Condensed*.ttf")))
        except Exception:
            pass
    for p in ttf_paths:
        try:
            fm.fontManager.addfont(p)
        except Exception:
            pass
    try: fm._rebuild()
    except Exception: pass
    mpl.rcParams["font.family"] = "Roboto Condensed"

use_roboto_condensed()

# -------------------------
# Style knobs
# -------------------------
FIGSIZE = (9, 6)
TITLE_SIZE = 25
TITLE_PAD  = 15
LABEL_SIZE = 22
TICK_SIZE  = 18
LEGEND_SIZE = 15
Y_TICK_STEP = 50
YMIN = 725

# Error bar styling (manual look)
ERR_WIDTH  = 2.0
CAP_WIDTH  = 8
DOT_SIZE   = 36

# --------- Helpers ---------
def p_to_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

def linregress_stats(x, y):
    """(kept for rare OLS fallback) Return slope, intercept, r, p, stderr, n, t, df, CI95."""
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 3 or np.unique(x).size < 2:
        return dict(slope=np.nan, intercept=np.nan, r=np.nan, p=np.nan,
                    stderr=np.nan, n=x.size, t=np.nan, df=np.nan,
                    ci_lo=np.nan, ci_hi=np.nan)
    res = stats.linregress(x, y)
    slope, intercept, r, p, se = res.slope, res.intercept, res.rvalue, res.pvalue, res.stderr
    n = x.size
    df = n - 2
    t = slope / se if np.isfinite(se) and se > 0 else np.nan
    tcrit = stats.t.ppf(0.975, df) if np.isfinite(df) and df > 0 else np.nan
    ci_lo = slope - tcrit * se if np.isfinite(tcrit) else np.nan
    ci_hi = slope + tcrit * se if np.isfinite(tcrit) else np.nan
    return dict(slope=slope, intercept=intercept, r=r, p=p, stderr=se, n=n, t=t, df=df, ci_lo=ci_lo, ci_hi=ci_hi)

def fit_mixed(df_in, x="true_count_Condition", y="SUM_partial", group="participant", re_formula="1"):
    """
    Mixed model with random intercept for 'group'.
    Returns intercept, slope (beta for x), z, p, CI95, plus a label string.
    Falls back to OLS if MixedLM can't run.
    """
    need = [x, y, group]
    if not all(k in df_in.columns for k in need):
        raise ValueError(f"Missing columns: {set(need) - set(df_in.columns)}")

    sub = df_in[need].dropna().copy()
    if sub[group].nunique() < 2 or sub[x].nunique() < 2:
        # fallback to OLS
        s = linregress_stats(sub[x], sub[y])
        return dict(
            intercept=s["intercept"], slope=s["slope"],
            se=s["stderr"], stat=s["t"], stat_name="t",
            p=s["p"], ci_lo=s["ci_lo"], ci_hi=s["ci_hi"],
            method="OLS"
        )

    # MixedLM
    sub[group] = sub[group].astype("category")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = smf.mixedlm(f"{y} ~ {x}", sub, groups=sub[group], re_formula=re_formula)
            res = model.fit(method="lbfgs", reml=True, maxiter=200, disp=False)
        beta = res.params.get(x, np.nan)
        intercept = res.params.get("Intercept", np.nan)
        se = res.bse.get(x, np.nan)
        stat = res.tvalues.get(x, np.nan)   # statsmodels MixedLM -> Wald z
        p = res.pvalues.get(x, np.nan)
        ci = res.conf_int()
        ci_lo, ci_hi = (ci.loc[x, 0], ci.loc[x, 1]) if x in ci.index else (np.nan, np.nan)
        return dict(
            intercept=intercept, slope=beta, se=se,
            stat=stat, stat_name="z", p=p, ci_lo=ci_lo, ci_hi=ci_hi,
            method="MixedLM"
        )
    except Exception:
        # Robust fallback: OLS
        s = linregress_stats(sub[x], sub[y])
        return dict(
            intercept=s["intercept"], slope=s["slope"],
            se=s["stderr"], stat=s["t"], stat_name="t",
            p=s["p"], ci_lo=s["ci_lo"], ci_hi=s["ci_hi"],
            method="OLS"
        )

def mixed_stats_text(stats_d):
    if not np.isfinite(stats_d.get("slope", np.nan)):
        return ""
    stat_name = stats_d.get("stat_name", "z")
    return f"(β = {stats_d['slope']:.3f}, {stat_name} = {stats_d['stat']:.2f}, p = {stats_d['p']:.3g})"

# Manual-style error bars
def draw_errorbar(ax, x, mean, se, top_color="black", bottom_color="black",
                  elinewidth=None, capsize=None, z_top=4, z_bottom=3):
    if not np.isfinite(se) or se <= 0:
        return
    if elinewidth is None: elinewidth = ERR_WIDTH
    if capsize is None:    capsize    = CAP_WIDTH
    eb_pos = ax.errorbar(x, mean, yerr=[[0],[se]], fmt="none",
                         ecolor=top_color, capsize=capsize, elinewidth=elinewidth, zorder=z_top)
    for cap in eb_pos[1]:
        cap.set_markeredgewidth(elinewidth); cap.set_markersize(capsize)
    eb_neg = ax.errorbar(x, mean, yerr=[[se],[0]], fmt="none",
                         ecolor=bottom_color, capsize=capsize, elinewidth=elinewidth, zorder=z_bottom)
    for cap in eb_neg[1]:
        cap.set_markeredgewidth(elinewidth); cap.set_markersize(capsize)

# Panel plot (now accepts stats_text and explicit slope/intercept)
def plot_panel(
    df_in, intercept=None, slope=None, title="", ylim=None,
    dot_color="black", line_color="pink", show_legend=True,
    title_size=TITLE_SIZE, title_pad=TITLE_PAD,
    label_size=LABEL_SIZE, tick_size=TICK_SIZE,
    legend_size=LEGEND_SIZE, y_tick_step=Y_TICK_STEP,
    x_label="true_count_Condition", y_label="Trial pair sum",
    include_stats_in_title=True, stats_text=None,
    err_top_color="black", err_bottom_color="black"   # <— NEW
):
    xcol, ycol = "true_count_Condition", "SUM_partial"
    ax = plt.gca()

    # Bin means ± SE
    gb = (df_in.groupby(xcol)[ycol].agg(["mean","count","std"]).reset_index())
    gb["se"] = gb["std"] / np.sqrt(gb["count"].clip(lower=1))

    ax.scatter(gb[xcol], gb["mean"], s=DOT_SIZE, color=dot_color, zorder=5, label="Bin mean")
    for xi, mu, se in zip(gb[xcol].to_numpy(), gb["mean"].to_numpy(), gb["se"].to_numpy()):
        draw_errorbar(ax, xi, mu, se,
                      top_color=err_top_color, bottom_color=err_bottom_color)

    # Fit line (use provided)
    x_min, x_max = df_in[xcol].min(), df_in[xcol].max()
    if np.isfinite(slope) and np.isfinite(intercept) and x_min < x_max:
        xgrid = np.linspace(x_min, x_max, 200)
        ax.plot(xgrid, intercept + slope * xgrid,
                color=line_color, linewidth=2.5, label="Mixed-effects fit", zorder=8)

    # Title
    suffix = f"  {stats_text}" if (include_stats_in_title and stats_text) else ""
    ax.set_title(f"{title}{suffix}", fontsize=title_size, pad=title_pad)
    ax.set_xlabel(x_label, fontsize=label_size)
    ax.set_ylabel(y_label, fontsize=label_size)
    if ylim is not None:
        ax.set_ylim(ylim)


    ax.yaxis.set_major_locator(mticker.MultipleLocator(y_tick_step))
    ax.tick_params(axis="both", labelsize=tick_size)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    if show_legend:
        proxy_dot  = Line2D([0], [0], marker='o', linestyle='None', color=dot_color, markersize=np.sqrt(DOT_SIZE))
        proxy_line = Line2D([0], [0], linestyle='-', color=line_color, linewidth=2.5)
        ax.legend([proxy_dot, proxy_line], ["Bin mean ± SE", "Mixed-effects fit"], fontsize=legend_size)

    plt.tight_layout()

# =========================
# PREP: ensure types
# =========================
# Expect df to already exist. Ensure required columns are present.
for col in ["true_count_Condition", "SUM_partial", "participant", "Condition"]:
    if col not in df.columns:
        raise ValueError(f"Required column '{col}' not found in df")

df["participant"] = df["participant"].astype("category")
df["Condition"] = df["Condition"].astype("category")

# =========================
# Fit Mixed Model(s)
# =========================
overall = fit_mixed(df)  # random intercept for participant
overall_text = mixed_stats_text(overall)

# Per-condition mixed fits (also random intercept for participant)
per_cond = {}
for c in ["SUP-SUB", "SUB-SUP"]:
    sub = df[df["Condition"] == c]
    per_cond[c] = fit_mixed(sub)
    per_cond[c]["text"] = mixed_stats_text(per_cond[c])

# =========================
# Figure 1: Full-range (mixed fit)
# =========================
plt.figure(figsize=FIGSIZE)
plot_panel(
    df,
    intercept=overall["intercept"], slope=overall["slope"],
    title="Means binned by number of relationship sharing neighbors",
    dot_color="black", line_color="#f348ba",
    x_label="Number of neighbors", y_label="Trial pair sum (ms)",
    include_stats_in_title=True, stats_text=overall_text
)
plt.show()

# =========================
# Figure 2: Zoomed (mixed fit)
# =========================
plt.figure(figsize=FIGSIZE)
plot_panel(
    df,
    intercept=overall["intercept"], slope=overall["slope"],
    title="Means binned by number of relationship sharing neighbors",
    ylim=(1375, 1575),
    dot_color="black", line_color="#fa6833",
    x_label="Number of neighbors", y_label="Trial pair sum (ms)",
    include_stats_in_title=True, stats_text=overall_text
)
plt.show()

# =========================
# Figures 3 & 4: Split by Condition (mixed fits per condition)
# =========================
for c in ["SUP-SUB", "SUB-SUP"]:
    subdf = df[df["Condition"] == c].copy()
    plt.figure(figsize=FIGSIZE)

    if subdf.empty:
        ax = plt.gca()
        ax.set_title(f"{c} (no data)", fontsize=TITLE_SIZE, pad=TITLE_PAD)
        ax.set_xlabel("Number of neighbors", fontsize=LABEL_SIZE)
        ax.set_ylabel("Trial pair sum (ms)", fontsize=LABEL_SIZE)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(Y_TICK_STEP))
        ax.tick_params(axis="both", labelsize=TICK_SIZE)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.show()
        continue

    # Dots = condition color; line = black
    dot_color  = "#f348ba" if c == "SUP-SUB" else "#01b0f0"  # dots
    line_color = "black"                                      # fit line

    plot_panel(
        subdf,
        intercept=per_cond[c]["intercept"], slope=per_cond[c]["slope"],
        title=f"{c}: Means binned by number of relationship sharing neighbors",
        ylim=(1350, 1625),
        dot_color=dot_color, line_color=line_color, show_legend=True,
        x_label="Number of neighbors", y_label="Trial pair sum (ms)",
        include_stats_in_title=False, stats_text=per_cond[c]["text"],
        err_top_color=dot_color, err_bottom_color=dot_color   # <— match dots
    )
    plt.show()

# =========================
# Figure 5 (polished): Combined — bin means ± SE by condition + mixed fits
# =========================
from matplotlib.legend_handler import HandlerErrorbar
import matplotlib as mpl

plt.figure(figsize=FIGSIZE)
ax = plt.gca()

xcol, ycol = "true_count_Condition", "SUM_partial"

# Colors
BLUE   = "#01b0f0"   # SUP-SUB
PINK   = "#f348ba"   # SUB-SUP
ORANGE = "#fa6833"   # overall

# Collect bin stats for y-limit calculation
bin_tables = []
for cond in ["SUP-SUB", "SUB-SUP"]:
    sub = df[df["Condition"] == cond]
    if sub.empty or sub[xcol].nunique() < 2:
        continue
    gb = (sub.groupby(xcol)[ycol]
            .agg(["mean","count","std"])
            .reset_index()
            .sort_values(xcol))
    gb["se"] = gb["std"] / np.sqrt(gb["count"].clip(lower=1))
    gb["Condition"] = cond
    bin_tables.append(gb)

# Plot each condition’s bin means + SE (colored)
for cond, color in [("SUP-SUB", BLUE), ("SUB-SUP", PINK)]:
    gb = next((t for t in bin_tables if not t.empty and t["Condition"].iloc[0] == cond), None)
    sub = df[df["Condition"] == cond]
    if gb is None or sub.empty:
        continue

    ax.scatter(
        gb[xcol], gb["mean"],
        s=DOT_SIZE, color=color, edgecolor="none", alpha=0.95,
        label=f"{cond} bin means", zorder=6
    )
    for xi, mu, se in zip(gb[xcol].to_numpy(), gb["mean"].to_numpy(), gb["se"].to_numpy()):
        draw_errorbar(
            ax, xi, mu, se,
            top_color=color, bottom_color=color,
            elinewidth=ERR_WIDTH, capsize=CAP_WIDTH,
            z_top=5, z_bottom=5
        )

    # Per-condition Mixed fit (fixed effects line)
    s = per_cond[cond]
    if np.isfinite(s["slope"]) and np.isfinite(s["intercept"]):
        xgrid = np.linspace(sub[xcol].min(), sub[xcol].max(), 200)
        ax.plot(
            xgrid, s["intercept"] + s["slope"] * xgrid,
            linestyle="--", linewidth=2.8, color=color,
            label=f"{cond} mixed fit (β {s['slope']:.3f}, {s['stat_name']} {s['stat']:.2f}, p {s['p']:.3g})",
            zorder=10
        )

# Overall mixed fit (solid orange, on top)
if np.isfinite(overall["slope"]) and np.isfinite(overall["intercept"]):
    xgrid_all = np.linspace(df[xcol].min(), df[xcol].max(), 300)
    ax.plot(
        xgrid_all, overall["intercept"] + overall["slope"] * xgrid_all,
        linestyle="-", linewidth=3.2, color=ORANGE,
        label=f"Overall mixed fit (β {overall['slope']:.3f}, {overall['stat_name']} {overall['stat']:.2f}, p {overall['p']:.3g})",
        zorder=11
    )

# Title/labels/ticks
ax.set_title("All fits on one plot (bin means ± SE; mixed-effects lines)", fontsize=TITLE_SIZE, pad=TITLE_PAD)
ax.set_xlabel("Number of neighbors", fontsize=LABEL_SIZE)
ax.set_ylabel("Trial pair sum (ms)", fontsize=LABEL_SIZE)
ax.tick_params(axis="both", labelsize=TICK_SIZE)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_locator(mticker.MultipleLocator(Y_TICK_STEP))

# Y-limits from bin means ± SE across both conditions
if bin_tables:
    all_bins = pd.concat(bin_tables, ignore_index=True)
    ymin = (all_bins["mean"] - all_bins["se"]).min()
    ymax = (all_bins["mean"] + all_bins["se"]).max()
    pad = 0.05 * (ymax - ymin if np.isfinite(ymax - ymin) and (ymax - ymin) > 0 else 1.0)
    lo = ymin - pad
    hi = ymax + pad
    if YMIN is not None:
        lo = max(lo, YMIN)
    ax.set_ylim(lo, hi)

# ------- Legend below figure -------
handles = []
labels  = []

eb_sup = ax.errorbar([], [], yerr=[[0.5], [0.5]], fmt='o',
                     mfc=BLUE, mec=BLUE, ecolor=BLUE, elinewidth=ERR_WIDTH,
                     capsize=CAP_WIDTH, markersize=np.sqrt(DOT_SIZE))
eb_sub = ax.errorbar([], [], yerr=[[0.5], [0.5]], fmt='o',
                     mfc=PINK, mec=PINK, ecolor=PINK, elinewidth=ERR_WIDTH,
                     capsize=CAP_WIDTH, markersize=np.sqrt(DOT_SIZE))

line_sup, = ax.plot([], [], linestyle="--", color=BLUE, linewidth=2.8)
line_sub, = ax.plot([], [], linestyle="--", color=PINK, linewidth=2.8)
line_all, = ax.plot([], [], linestyle="-",  color=ORANGE, linewidth=3.2)

handles.extend([eb_sup, eb_sub, line_sup, line_sub, line_all])
labels.extend (["SUP-SUB bin mean ± SE", "SUB-SUP bin mean ± SE",
                "SUP-SUB mixed fit", "SUB-SUP mixed fit", "Overall mixed fit"])

legend = ax.legend(
    handles, labels,
    handler_map={mpl.container.ErrorbarContainer: HandlerErrorbar(xerr_size=0.0, yerr_size=1.2)},
    fontsize=LEGEND_SIZE, frameon=False, ncol=2,
    loc="upper center", bbox_to_anchor=(0.5, -0.18), borderaxespad=0.0, handlelength=3
)

plt.tight_layout()
plt.show()

# =========================
# Mixed-Effects summary table (global + per-condition)
# =========================
def summarize_mixed(df_in, label):
    s = fit_mixed(df_in)
    return {
        "Group": label,
        "beta": s["slope"],
        "Intercept": s["intercept"],
        s["stat_name"]: s["stat"],
        "p": s["p"],
        "SE_beta": s["se"],
        "CI95_lo": s["ci_lo"],
        "CI95_hi": s["ci_hi"],
        "sig": p_to_stars(s["p"]) if np.isfinite(s["p"]) else "",
        "Method": s["method"]
    }

rows = [summarize_mixed(df, "ALL")]
for c in ["SUP-SUB", "SUB-SUP"]:
    subdf = df[df["Condition"] == c]
    if not subdf.empty:
        rows.append(summarize_mixed(subdf, c))

mix_summary = pd.DataFrame(rows)

with pd.option_context('display.max_columns', None, 'display.width', 120):
    print("\nMixed-effects summary (y = SUM_partial, x = true_count_Condition; random intercept = participant)")
    print("(β is the fixed-effect slope; stats are Wald z unless OLS fallback)\n")
    def fmt(x, nd=3):
        return "nan" if (x is None or not np.isfinite(x)) else f"{x:.{nd}f}"
    # Pretty-print with flexible stat column name
    stat_col = "z" if "z" in mix_summary.columns else "t"
    print(mix_summary.rename(columns={"beta":"β", "SE_beta":"SE(β)"}).to_string(
        index=False,
        formatters={
            "β":           lambda v: fmt(v, 4),
            "Intercept":   lambda v: fmt(v, 2),
            stat_col:      lambda v: fmt(v, 2),
            "p":           lambda v: ("nan" if not np.isfinite(v) else f"{v:.3g}"),
            "SE(β)":       lambda v: fmt(v, 4),
            "CI95_lo":     lambda v: fmt(v, 4),
            "CI95_hi":     lambda v: fmt(v, 4),
        }
    ))
    # =========================
# Non-binned (RAW) OLS and BIN-MEANS weighted OLS summaries
# =========================

def fit_ols_trial(df_in, x="true_count_Condition", y="SUM_partial"):
    """Plain OLS on all trials (non-binned). Uses scipy.linregress for t/CI."""
    sub = df_in[[x, y]].dropna()
    if sub.shape[0] < 3 or sub[x].nunique() < 2:
        return dict(intercept=np.nan, slope=np.nan, se=np.nan, stat=np.nan, stat_name="t",
                    p=np.nan, ci_lo=np.nan, ci_hi=np.nan, r=np.nan, r2=np.nan, df=np.nan,
                    method="OLS-raw")
    s = linregress_stats(sub[x], sub[y])
    return dict(
        intercept=s["intercept"], slope=s["slope"], se=s["stderr"],
        stat=s["t"], stat_name="t", p=s["p"], ci_lo=s["ci_lo"], ci_hi=s["ci_hi"],
        r=s["r"], r2=(s["r"]**2 if np.isfinite(s["r"]) else np.nan),
        df=s["df"], method="OLS-raw"
    )

def fit_bin_weighted_ols(df_in, x="true_count_Condition", y="SUM_partial"):
    """
    Weighted OLS on BIN MEANS: y_mean ~ x, weights = bin counts.
    Matches what you'd report as 'binned means (weighted)'.
    """
    sub = df_in[[x, y]].dropna()
    gb = (sub.groupby(x)[y].agg(mean="mean", count="count").reset_index())
    if gb.shape[0] < 3 or gb[x].nunique() < 2:
        return dict(intercept=np.nan, slope=np.nan, se=np.nan, stat=np.nan, stat_name="t",
                    p=np.nan, ci_lo=np.nan, ci_hi=np.nan, r2=np.nan, df=np.nan,
                    method="WLS-bin")
    # Use named columns to keep parameter names in results
    X = pd.DataFrame({"const": 1.0, x: gb[x].to_numpy()})
    w = gb["count"].to_numpy()
    y_mean = gb["mean"].to_numpy()
    res = sm.WLS(y_mean, X, weights=w).fit()
    # Conf int is a DataFrame indexed by 'const' and x
    ci = res.conf_int()
    ci_lo, ci_hi = (ci.loc[x, 0], ci.loc[x, 1]) if x in ci.index else (np.nan, np.nan)
    return dict(
        intercept=res.params.get("const", np.nan),
        slope=res.params.get(x, np.nan),
        se=res.bse.get(x, np.nan),
        stat=res.tvalues.get(x, np.nan),
        stat_name="t",
        p=res.pvalues.get(x, np.nan),
        ci_lo=ci_lo, ci_hi=ci_hi,
        r2=res.rsquared if hasattr(res, "rsquared") else np.nan,
        df=float(res.df_resid) if hasattr(res, "df_resid") else np.nan,
        method="WLS-bin"
    )

def summarize_generic(df_in, label, fitter, include_r=False):
    s = fitter(df_in)
    row = {
        "Group": label,
        "β": s["slope"],
        "Intercept": s["intercept"],
        "t": s["stat"],
        "p": s["p"],
        "SE(β)": s["se"],
        "CI95_lo": s["ci_lo"],
        "CI95_hi": s["ci_hi"],
        "df": s.get("df", np.nan),
        "R²": s.get("r2", np.nan),
        "sig": p_to_stars(s["p"]) if np.isfinite(s["p"]) else "",
        "Method": s.get("method", "")
    }
    if include_r:
        row["r"] = s.get("r", np.nan)
    return row

# ---------- Build tables ----------
groups = [("ALL", df), ("SUP-SUB", df[df["Condition"] == "SUP-SUB"]), ("SUB-SUP", df[df["Condition"] == "SUB-SUP"])]

ols_rows = [summarize_generic(d, label, fit_ols_trial, include_r=True) for (label, d) in groups if not d.empty]
bin_rows = [summarize_generic(d, label, fit_bin_weighted_ols, include_r=False) for (label, d) in groups if not d.empty]

ols_summary = pd.DataFrame(ols_rows)
bin_summary = pd.DataFrame(bin_rows)

# ---------- Pretty print ----------
def _fmt(x, nd=3):
    return "nan" if (x is None or not np.isfinite(x)) else f"{x:.{nd}f}"

print("\nRAW OLS summary (trial-level; y = SUM_partial, x = true_count_Condition)\n")
print(ols_summary.to_string(
    index=False,
    formatters={
        "β":         lambda v: _fmt(v, 4),
        "Intercept": lambda v: _fmt(v, 2),
        "t":         lambda v: _fmt(v, 2),
        "p":         lambda v: "nan" if not np.isfinite(v) else f"{v:.3g}",
        "SE(β)":     lambda v: _fmt(v, 4),
        "CI95_lo":   lambda v: _fmt(v, 4),
        "CI95_hi":   lambda v: _fmt(v, 4),
        "df":        lambda v: "nan" if not np.isfinite(v) else f"{int(round(v))}",
        "r":         lambda v: _fmt(v, 3),
        "R²":        lambda v: _fmt(v, 3),
    }
))

print("\nBIN-MEANS (weighted OLS) summary (means ±SE per neighbor; weights = bin counts)\n")
print(bin_summary.to_string(
    index=False,
    formatters={
        "β":         lambda v: _fmt(v, 4),
        "Intercept": lambda v: _fmt(v, 2),
        "t":         lambda v: _fmt(v, 2),
        "p":         lambda v: "nan" if not np.isfinite(v) else f"{v:.3g}",
        "SE(β)":     lambda v: _fmt(v, 4),
        "CI95_lo":   lambda v: _fmt(v, 4),
        "CI95_hi":   lambda v: _fmt(v, 4),
        "df":        lambda v: "nan" if not np.isfinite(v) else f"{int(round(v))}",
        "R²":        lambda v: _fmt(v, 3),
    }
))

    


In [ ]:
from scipy import stats
import numpy as np
import pandas as pd

# --- Helpers ---
def p_to_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

def linregress_stats(x, y, weights=None):
    """
    Return slope, intercept, r, p, stderr, n, t, df, 95% CI.
    If weights is None, use scipy.stats.linregress (OLS).
    If weights provided, do a WLS-equivalent closed form and a weighted Pearson r.
    """
    x = np.asarray(x); y = np.asarray(y)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if weights is not None:
        w = np.asarray(weights)[ok]
    n = x.size
    if n < 3 or np.unique(x).size < 2:
        return dict(slope=np.nan, intercept=np.nan, r=np.nan, p=np.nan,
                    stderr=np.nan, n=n, t=np.nan, df=np.nan,
                    ci_lo=np.nan, ci_hi=np.nan, method="WLS" if weights is not None else "OLS")

    if weights is None:
        res = stats.linregress(x, y)  # OLS
        slope, intercept, r, p, se = res.slope, res.intercept, res.rvalue, res.pvalue, res.stderr
        df = n - 2
        t = slope / se if np.isfinite(se) and se > 0 else np.nan
        tcrit = stats.t.ppf(0.975, df) if df > 0 else np.nan
        ci_lo = slope - tcrit * se if np.isfinite(tcrit) else np.nan
        ci_hi = slope + tcrit * se if np.isfinite(tcrit) else np.nan
        return dict(slope=slope, intercept=intercept, r=r, p=p, stderr=se,
                    n=n, t=t, df=df, ci_lo=ci_lo, ci_hi=ci_hi, method="OLS")
    else:
        # Weighted means
        w = w.astype(float)
        W = w.sum()
        mx = (w * x).sum() / W
        my = (w * y).sum() / W
        # Weighted covariance/variance
        cov_xy = (w * (x - mx) * (y - my)).sum() / W
        var_x  = (w * (x - mx)**2).sum() / W
        var_y  = (w * (y - my)**2).sum() / W
        slope = cov_xy / var_x
        intercept = my - slope * mx
        # Weighted Pearson r
        r = cov_xy / np.sqrt(var_x * var_y)
        # SE for slope under WLS (approx; uses effective dof)
        # Compute weighted residuals
        yhat = intercept + slope * x
        resid = y - yhat
        # Effective df ≈ number of unique x minus 2
        df = max(int(np.unique(x).size) - 2, 1)
        s2 = (w * resid**2).sum() / (W - 2) if W > 2 else np.nan
        stderr = np.sqrt(s2 / (W * var_x)) if np.isfinite(s2) and var_x > 0 and W > 0 else np.nan
        t = slope / stderr if np.isfinite(stderr) and stderr > 0 else np.nan
        p = 2 * stats.t.sf(abs(t), df) if np.isfinite(t) and df > 0 else np.nan
        tcrit = stats.t.ppf(0.975, df) if df > 0 else np.nan
        ci_lo = slope - tcrit * stderr if np.isfinite(tcrit) else np.nan
        ci_hi = slope + tcrit * stderr if np.isfinite(tcrit) else np.nan
        return dict(slope=slope, intercept=intercept, r=r, p=p, stderr=stderr,
                    n=n, t=t, df=df, ci_lo=ci_lo, ci_hi=ci_hi, method="WLS")

def summarize_block(df_in, label):
    xcol, ycol = "true_count_Condition", "SUM_partial"
    # 1) Raw trials (what gave you r ~ -0.028)
    s_raw = linregress_stats(df_in[xcol], df_in[ycol])

    # 2) Bin means (unweighted)
    gb = df_in.groupby(xcol)[ycol].agg(["mean","count"]).reset_index()
    s_bin_unw = linregress_stats(gb[xcol], gb["mean"])

    # 3) Bin means (weighted by bin size)
    s_bin_w = linregress_stats(gb[xcol], gb["mean"], weights=gb["count"])

    def row(s, kind):
        return {
            "Group": label,
            "Fit": kind,
            "n_points": s["n"],
            "slope": s["slope"],
            "r": s["r"],
            "r2": (s["r"]**2 if np.isfinite(s["r"]) else np.nan),
            "t": s["t"],
            "df": s["df"],
            "p": s["p"],
            "stderr": s["stderr"],
            "CI95_lo": s["ci_lo"],
            "CI95_hi": s["ci_hi"],
            "sig": p_to_stars(s["p"]) if np.isfinite(s["p"]) else "",
        }

    return [row(s_raw, "RAW (OLS)"),
            row(s_bin_unw, "BIN MEANS (unweighted)"),
            row(s_bin_w,   "BIN MEANS (weighted)")]

# ---------- Build the comparison table ----------
rows = []
rows += summarize_block(df, "ALL")
for c in ["SUP-SUB", "SUB-SUP"]:
    subdf = df[df["Condition"] == c]
    if not subdf.empty:
        rows += summarize_block(subdf, c)

compare = pd.DataFrame(rows)

# Pretty print
with pd.option_context('display.max_columns', None, 'display.width', 140):
    print("\nCorrelation/regression comparison across RAW vs BIN MEANS")
    print("(Slope units: ΔSUM per +1 neighbor)\n")
    print(compare.to_string(index=False, formatters={
        "slope":   "{:.4f}".format,
        "r":       "{:.3f}".format,
        "r2":      "{:.4f}".format,
        "t":       "{:.2f}".format,
        "df":      "{:.0f}".format,
        "p":       "{:.3g}".format,
        "stderr":  "{:.4f}".format,
        "CI95_lo": "{:.4f}".format,
        "CI95_hi": "{:.4f}".format
    }))
